In [1]:
import os
from dotenv import load_dotenv

## langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS

/var/folders/0b/sllx3hsj4sgcy1gc4s3nf7980000gn/T/ipykernel_65287/1899740434.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

In [3]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("Environment variable loaded")

Environment variable loaded


LOADING OUR DATA

In [4]:
DATA_FILE_PATH = os.path.join("data","hr_policy.txt")
print(DATA_FILE_PATH)

data/hr_policy.txt


#### DATA INGESTION

In [5]:
loader = TextLoader(DATA_FILE_PATH,encoding="utf-8")

documents = loader.load()
print("DATA LOADED ")
print("="*40)
print(documents)


DATA LOADED 
[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

LANGCHAIN DOCUMENT

Langchain processes everything in form of documents

DOCUMENTS:

PAGE CONTENT - the actual data
METADATA - extra information about the data

In [6]:
len(documents)

1

In [7]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [8]:
print(documents[0].metadata)

{'source': 'data/hr_policy.txt'}


In [9]:
print(f"Total number of character in documents: {len(documents[0].page_content)}")

Total number of character in documents: 2597


SPLITTING OUR DATA

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)
chunks = text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to

In [11]:
len(chunks)

9

NOW EACH SPLITED CHUNK IS A DOCUMENT - a page content and metadata

In [12]:
print(chunks[1])

page_content='1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.' metadata={'source': 'data/hr_policy.txt'}


In [13]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


EMBEDD OUR DATA

In [14]:
embeddings_model = JinaEmbeddings(model_name = "jina-embeddings-v2-base-en")
print("EMB MODEL READY THE NAME IS",embeddings_model.model_name)

EMB MODEL READY THE NAME IS jina-embeddings-v2-base-en


STORE DATA IN VECTOR DATABASE

In [15]:
vector_store = FAISS.from_documents(chunks,embeddings_model)
print("CHUNKS ARE STORED",vector_store.index.ntotal)

CHUNKS ARE STORED 9


WE NEVER STORED IT

In [16]:
test_query = "how many sick leaves do employee get?"

## similarity search

top_matches = vector_store.similarity_search(test_query,k=2)
print(top_matches)

for i,match in enumerate(top_matches,start = 1):
    print(match.page_content)
    print()

[Document(id='85844a72-b6ac-4580-926b-8dff033bd87d', metadata={'source': 'data/hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(id='5477edee-375e-4e52-bf87-aaac33712105', metadata={'source': 'data/hr_policy.txt'}, page_content='7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.')]
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calenda

In [43]:
## TOOL
retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question :str)->str:
    """
    search the HR policy document for information about leave,
    work from home, probation, notice period, reimbursemet, code 
    of conducts, holidays, or exit process
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

DATA RETRIEVAL

LLM

In [17]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0
)

llm.model_name

'openai/gpt-oss-120b'

In [18]:
test_response = llm.invoke("Hey is learning rag is hard? reply in a joke manner")


In [19]:
print(test_response.content)

Learning RAG? Nah—it's only as hard as trying to teach a cat to fetch the newspaper. You’ll spend a lot of time convincing the model “no, we don’t need that 1998‑year‑old meme,” but eventually it’ll fetch the right info—just don’t expect it to bring you a latte on the way! 😸📚🚀


AI AGENT
-- ai agents have three things

1.LLM - BRAIN
2.TOOL - EXTRA SUPER POWER
3.MEMORY 


In [20]:
from langchain.agents import create_agent

In [49]:
hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt= 
    """
You are a friendly HR assistant working for Acme Crop.Always use the search_hr_policy tool 
to look up the facts before answering.If the answer is not in the search 
results,say you do not know instead of guessing.
"""
)
print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [50]:
def ask_hr_assistant(question : str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" *60)
    print("QUESTION : ", question)
    print("-" * 60)

    response = hr_assistant.invoke(
        {
            "messages": [
                {
                    "role":"user",
                    "content":question
                }
            ]
        }
    )


In [52]:
response = hr_assistant.invoke(
        {
            "messages": [
                {
                    "role":"user",
                    "content":"for which organization are you working?"
                }
            ]
        }
    )


In [53]:
response

{'messages': [HumanMessage(content='for which organization are you working?', additional_kwargs={}, response_metadata={}, id='4170f3a2-4521-4387-aae4-691a9e0c6c52'),
  AIMessage(content='I’m the HR assistant for **Acme Crop**. How can I help you today?', additional_kwargs={'reasoning_content': 'The user asks: "for which organization are you working?" According to role, we are a friendly HR assistant working for Acme Crop. We should answer based on that. No need to search policy. Just answer.'}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 205, 'total_tokens': 278, 'completion_time': 0.155660157, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.008910583, 'prompt_tokens_details': None, 'queue_time': 0.211771512, 'total_time': 0.16457074}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_19b184c447', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1c5ce-2

In [54]:
print(response["messages"][-1].content)

I’m the HR assistant for **Acme Crop**. How can I help you today?
